In [2]:
import gzip
import os

GZ_PATH = "../data_raw/discogs/discogs_20260401_releases.xml.gz"

# ── File size ────────────────────────────────────────────────────
size_gb = os.path.getsize(GZ_PATH) / (1024**3)
print(f"File size on disk: {size_gb:.2f} GB")

# ── Check if download is still running ───────────────────────────
# If the file is still growing, wait for it to finish first
import time
size1 = os.path.getsize(GZ_PATH)
time.sleep(3)
size2 = os.path.getsize(GZ_PATH)

if size2 > size1:
    grew_mb = (size2 - size1) / (1024**2)
    print(f"⏳ File is still downloading (+{grew_mb:.1f} MB in 3 sec)")
    print("   Wait for PyCharm download to finish, then re-run this cell.")
else:
    print("✅ File size is stable — download appears complete")

    # ── Verify gzip integrity ─────────────────────────────────────
    print("\nVerifying file integrity ...")
    count = 0
    try:
        with gzip.open(GZ_PATH, "rt", encoding="utf-8", errors="replace") as f:
            for line in f:
                if "<release " in line:
                    count += 1
                    print(f"  Release {count}: {line.strip()[:80]}")
                if count >= 5:
                    break
        print(f"\n✅ File is valid — ready for Step 2 (parse)")
    except Exception as e:
        print(f"❌ File corrupted: {e}")
        print("   Delete it and re-run 01_download_discogs.py")

File size on disk: 10.24 GB
✅ File size is stable — download appears complete

Verifying file integrity ...
  Release 1: <release id="1"><artists><artist><id>1</id><name>The Persuader</name></artist></
  Release 2: <release id="2"><artists><artist><id>2</id><name>Mr. James Barth &amp; A.D.</nam
  Release 3: <release id="3"><artists><artist><id>3</id><name>Josh Wink</name></artist></arti
  Release 4: <release id="4"><artists><artist><id>21</id><name>Faze Action</name></artist></a
  Release 5: <release id="5"><artists><artist><id>22</id><name>Datacide</name></artist></arti

✅ File is valid — ready for Step 2 (parse)


In [3]:
import duckdb
con = duckdb.connect("../data_processed/thesis.duckdb")
con.execute("CREATE OR REPLACE TABLE _test (x INTEGER)")
con.execute("INSERT INTO _test VALUES (1)")
n = con.execute("SELECT COUNT(*) FROM _test").fetchone()[0]
print(f"DuckDB write test: {n} row written ✅" if n == 1 else "❌ DuckDB write failed")
con.execute("DROP TABLE _test")
con.close()

DuckDB write test: 1 row written ✅


In [4]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")

# ── Confirm table loaded correctly ───────────────────────────────
n = con.execute(
    "SELECT COUNT(*) FROM discogs_releases_raw"
).fetchone()[0]
print(f"✅ discogs_releases_raw: {n:,} rows")

# ── Genre distribution ───────────────────────────────────────────
print("\n── Top genres in Discogs ──")
print(con.execute("""
    SELECT discogs_genre, COUNT(*) AS n
    FROM discogs_releases_raw
    WHERE discogs_genre IS NOT NULL
    GROUP BY discogs_genre
    ORDER BY n DESC
    LIMIT 15
""").fetchdf().to_string(index=False))

# ── Top styles ───────────────────────────────────────────────────
print("\n── Top styles in Discogs ──")
print(con.execute("""
    SELECT discogs_style, COUNT(*) AS n
    FROM discogs_releases_raw
    WHERE discogs_style IS NOT NULL
    GROUP BY discogs_style
    ORDER BY n DESC
    LIMIT 15
""").fetchdf().to_string(index=False))

# ── Country distribution ─────────────────────────────────────────
print("\n── Top countries ──")
print(con.execute("""
    SELECT discogs_country, COUNT(*) AS n
    FROM discogs_releases_raw
    WHERE discogs_country IS NOT NULL
    GROUP BY discogs_country
    ORDER BY n DESC
    LIMIT 10
""").fetchdf().to_string(index=False))

# ── Format distribution ──────────────────────────────────────────
print("\n── Release formats ──")
print(con.execute("""
    SELECT discogs_format, COUNT(*) AS n
    FROM discogs_releases_raw
    WHERE discogs_format IS NOT NULL
    GROUP BY discogs_format
    ORDER BY n DESC
    LIMIT 10
""").fetchdf().to_string(index=False))

✅ discogs_releases_raw: 19,035,253 rows

── Top genres in Discogs ──
         discogs_genre       n
                  Rock 5454580
            Electronic 4889274
                   Pop 1772560
                  Jazz 1397412
Folk, World, & Country 1394836
             Classical 1028829
               Hip Hop  761374
           Funk / Soul  683809
                 Latin  622754
                Reggae  345289
             Non-Music  273934
        Stage & Screen  142857
                 Blues  138326
            Children's   85872
      Brass & Military   43529

── Top styles in Discogs ──
   discogs_style      n
        Pop Rock 537586
           House 533221
Alternative Rock 435548
           Vocal 388824
         Country 332620
            Punk 308002
          Techno 306476
            Folk 276907
        Hardcore 272582
            Soul 265876
       Hard Rock 259675
         Chanson 252417
       Synth-pop 239126
      Indie Rock 221723
        Romantic 219282

── Top countries ──
d

In [5]:
#Normalise Discogs keys
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ── Build normalised matching keys ───────────────────────────────
# Same normalisation as CDS and WebTables:
# lowercase + remove non-alphanumeric + trim
print("Building discogs_cleaned_for_matching ...")

con.execute("""
    CREATE OR REPLACE TABLE discogs_cleaned_for_matching AS
    SELECT
        discogs_id,
        discogs_artist,
        discogs_title,
        discogs_year,
        discogs_country,
        discogs_genre,
        discogs_style,
        discogs_label,
        discogs_catno,
        discogs_format,

        -- Normalised artist key
        trim(regexp_replace(
            lower(coalesce(discogs_artist, '')),
            '[^a-z0-9]+', ' ', 'g'
        )) AS discogs_artist_key,

        -- Normalised title key
        trim(regexp_replace(
            lower(coalesce(discogs_title, '')),
            '[^a-z0-9]+', ' ', 'g'
        )) AS discogs_title_key

    FROM discogs_releases_raw
    WHERE
        -- Minimum key length to avoid short spurious matches
        length(trim(regexp_replace(
            lower(coalesce(discogs_artist, '')),
            '[^a-z0-9]+', ' ', 'g'
        ))) >= 2
        AND
        length(trim(regexp_replace(
            lower(coalesce(discogs_title, '')),
            '[^a-z0-9]+', ' ', 'g'
        ))) >= 2
""")

n = con.execute(
    "SELECT COUNT(*) FROM discogs_cleaned_for_matching"
).fetchone()[0]
print(f"✅ discogs_cleaned_for_matching: {n:,} rows")

# ── Quick check ──────────────────────────────────────────────────
print("\n── Sample normalised keys ──")
print(con.execute("""
    SELECT
        discogs_artist,
        discogs_artist_key,
        discogs_title,
        discogs_title_key,
        discogs_genre,
        discogs_style
    FROM discogs_cleaned_for_matching
    LIMIT 10
""").fetchdf().to_string(index=False))

Connected ✅
Building discogs_cleaned_for_matching ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ discogs_cleaned_for_matching: 18,356,215 rows

── Sample normalised keys ──
        discogs_artist discogs_artist_key               discogs_title        discogs_title_key discogs_genre discogs_style
         The Persuader      the persuader                   Stockholm                stockholm    Electronic    Deep House
Mr. James Barth & A.D. mr james barth a d Knockin' Boots (Vol 2 Of 2) knockin boots vol 2 of 2    Electronic   Broken Beat
             Josh Wink          josh wink      Profound Sounds Vol. 1    profound sounds vol 1    Electronic        Techno
           Faze Action        faze action               Moving Cities            moving cities    Electronic         House
              Datacide           datacide                  Flowerhead               flowerhead    Electronic      Abstract
Mr. James Barth & A.D. mr james barth a d Knockin' Boots (Vol 1 Of 2) knockin boots vol 1 of 2    Electronic        Techno
          Moonchildren       moonchildren             Moonchi

In [7]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")

# ── Memory optimisation settings ─────────────────────────────────
con.execute("SET threads = 2")                    # fewer threads = less RAM
con.execute("SET memory_limit = '8GB'")           # explicit memory cap
con.execute("SET preserve_insertion_order = false") # saves RAM during joins

print("Memory settings applied ✅")

# ── Build MB keys as a PERSISTENT TABLE first ────────────────────
# Instead of computing MB keys inside the CTE (recomputed in RAM),
# we materialise them to disk first — much more memory efficient
print("Building mb_keys_for_discogs (materialised to disk) ...")

con.execute("""
    CREATE OR REPLACE TABLE mb_keys_for_discogs AS
    SELECT
        release_id,
        COALESCE(main_artist_name, external_artist) AS mb_artist,
        COALESCE(release_title,    external_title)  AS mb_title,
        barcode,
        mb_year,
        total_tracks,
        is_linked_to_musicbrainz,
        trim(regexp_replace(
            lower(coalesce(
                COALESCE(main_artist_name, external_artist), '')),
            '[^a-z0-9]+', ' ', 'g'
        )) AS mb_artist_key,
        trim(regexp_replace(
            lower(coalesce(
                COALESCE(release_title, external_title), '')),
            '[^a-z0-9]+', ' ', 'g'
        )) AS mb_title_key
    FROM music_enrichment_base_for_webtables_v1
    WHERE release_id IS NOT NULL
      AND COALESCE(main_artist_name, external_artist) IS NOT NULL
      AND COALESCE(release_title,    external_title)  IS NOT NULL
""")

n_mb = con.execute(
    "SELECT COUNT(*) FROM mb_keys_for_discogs"
).fetchone()[0]
print(f"✅ mb_keys_for_discogs: {n_mb:,} rows")

# ── Build index on keys for fast join ────────────────────────────
print("Creating indexes for fast join ...")
con.execute("""
    CREATE INDEX IF NOT EXISTS idx_mb_keys
    ON mb_keys_for_discogs (mb_artist_key, mb_title_key)
""")
con.execute("""
    CREATE INDEX IF NOT EXISTS idx_discogs_keys
    ON discogs_cleaned_for_matching (discogs_artist_key, discogs_title_key)
""")
print("✅ Indexes created")

Memory settings applied ✅
Building mb_keys_for_discogs (materialised to disk) ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ mb_keys_for_discogs: 5,309,878 rows
Creating indexes for fast join ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Indexes created


In [1]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ════════════════════════════════════════════════════════════════
# Build enrichment delta — ONE best Discogs record per MB release
#
# Priority for "best" Discogs record:
#   1. Has style (more specific than genre)
#   2. Has country
#   3. Year closest to MB year
#   4. Lowest discogs_id (oldest/most established entry)
# ════════════════════════════════════════════════════════════════
print("Building discogs_mb_enrichment_delta ...")
print("(One best Discogs record per unique MB release_id)\n")

con.execute("""
    CREATE OR REPLACE TABLE discogs_mb_enrichment_delta AS
    WITH ranked AS (
        SELECT *,
            ROW_NUMBER() OVER (
                PARTITION BY release_id
                ORDER BY
                    -- Prefer entries with style data
                    CASE WHEN discogs_style   IS NOT NULL THEN 0 ELSE 1 END,
                    -- Prefer entries with country
                    CASE WHEN discogs_country IS NOT NULL THEN 0 ELSE 1 END,
                    -- Prefer entries with year
                    CASE WHEN discogs_year    IS NOT NULL THEN 0 ELSE 1 END,
                    -- Oldest Discogs entry first (lowest id)
                    discogs_id ASC
            ) AS rn
        FROM discogs_mb_exact_matches
        WHERE release_id IS NOT NULL
    )
    SELECT
        release_id,
        discogs_id,
        discogs_artist,
        discogs_title,
        discogs_genre,
        discogs_style,
        discogs_label,
        discogs_catno,
        discogs_format,
        discogs_country,
        discogs_year,
        match_score,
        match_type
    FROM ranked
    WHERE rn = 1
""")

n_delta = con.execute(
    "SELECT COUNT(*) FROM discogs_mb_enrichment_delta"
).fetchone()[0]
print(f"✅ discogs_mb_enrichment_delta: {n_delta:,} unique MB releases")

# ── Field coverage ────────────────────────────────────────────────
print("\n── Field coverage after dedup ──")
print(con.execute("""
    SELECT
        COUNT(*)                                    AS total,
        COUNT(discogs_genre)                        AS has_genre,
        COUNT(discogs_style)                        AS has_style,
        COUNT(discogs_label)                        AS has_label,
        COUNT(discogs_country)                      AS has_country,
        COUNT(discogs_year)                         AS has_year,
        COUNT(discogs_format)                       AS has_format,
        COUNT(discogs_catno)                        AS has_catno
    FROM discogs_mb_enrichment_delta
""").fetchdf().T.rename(columns={0: "count"}).to_string())

# ── Genre distribution ─────────────────────────────────────────
print("\n── Top genres added to MusicBrainz ──")
print(con.execute("""
    SELECT discogs_genre, COUNT(*) AS n
    FROM discogs_mb_enrichment_delta
    WHERE discogs_genre IS NOT NULL
    GROUP BY discogs_genre
    ORDER BY n DESC
""").fetchdf().to_string(index=False))

# ── 3-source corroboration ────────────────────────────────────────
print("\n── 3-source corroboration (CDS + WDC + Discogs) ──")
print(con.execute("""
    SELECT
        COUNT(*) AS discogs_enriched,
        COUNT(CASE WHEN is_linked_to_musicbrainz = TRUE
                   THEN 1 END) AS also_in_cds,
        COUNT(CASE WHEN w.release_id IS NOT NULL
                   THEN 1 END) AS also_in_wdc,
        COUNT(CASE WHEN is_linked_to_musicbrainz = TRUE
                    AND w.release_id IS NOT NULL
                   THEN 1 END) AS in_all_3_sources
    FROM discogs_mb_enrichment_delta d
    JOIN music_enrichment_base_for_webtables_v1 b
      ON d.release_id = b.release_id
    LEFT JOIN wdc_mb_enrichment_delta w
      ON d.release_id = w.release_id
""").fetchdf().to_string(index=False))

# ── Final thesis numbers ──────────────────────────────────────────
n_mb     = con.execute("""
    SELECT COUNT(*) FROM music_enrichment_base_for_webtables_v1
    WHERE release_id IS NOT NULL
""").fetchone()[0]
n_cds    = con.execute("""
    SELECT COUNT(*) FROM music_enrichment_base_for_webtables_v1
    WHERE is_linked_to_musicbrainz = TRUE
""").fetchone()[0]
n_wdc    = con.execute(
    "SELECT COUNT(DISTINCT release_id) FROM wdc_mb_enrichment_delta"
).fetchone()[0]
n_disc   = con.execute(
    "SELECT COUNT(DISTINCT release_id) FROM discogs_mb_enrichment_delta"
).fetchone()[0]
n_any    = con.execute("""
    SELECT COUNT(DISTINCT release_id)
    FROM (
        SELECT release_id FROM music_enrichment_base_for_webtables_v1
        WHERE is_linked_to_musicbrainz = TRUE
          AND release_id IS NOT NULL
        UNION
        SELECT release_id FROM wdc_mb_enrichment_delta
        UNION
        SELECT release_id FROM discogs_mb_enrichment_delta
    )
""").fetchone()[0]

print(f"""
╔══════════════════════════════════════════════════════════════╗
║   FINAL THESIS REPORT — 3-SOURCE ENRICHMENT                 ║
╠══════════════════════════════════════════════════════════════╣
║  MusicBrainz total releases    : {n_mb:>10,}               ║
║  ──────────────────────────────────────────────────          ║
║  Phase 1 — CDS                 : {n_cds:>10,}  ({100*n_cds/n_mb:.2f}%)  ║
║  Phase 2 — WebTables           : {n_wdc:>10,}  ({100*n_wdc/n_mb:.2f}%)  ║
║  Phase 3 — Discogs             : {n_disc:>10,}  ({100*n_disc/n_mb:.2f}%)  ║
║  ──────────────────────────────────────────────────          ║
║  Enriched by at least 1 source : {n_any:>10,}  ({100*n_any/n_mb:.2f}%)  ║
╚══════════════════════════════════════════════════════════════╝
""")

# ── Sample — clean deduplicated results ───────────────────────────
print("── Sample: one best Discogs record per MB release ──")
display(con.execute("""
    SELECT
        discogs_artist,
        discogs_title,
        discogs_genre,
        discogs_style,
        discogs_country,
        discogs_year,
        discogs_label,
        release_id
    FROM discogs_mb_enrichment_delta
    WHERE discogs_style IS NOT NULL
    ORDER BY release_id
    LIMIT 15
""").fetchdf())

Connected ✅
Building discogs_mb_enrichment_delta ...
(One best Discogs record per unique MB release_id)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ discogs_mb_enrichment_delta: 1,608,595 unique MB releases

── Field coverage after dedup ──
               count
total        1608595
has_genre    1608595
has_style    1436481
has_label    1608595
has_country  1555626
has_year     1579156
has_format   1608595
has_catno    1608592

── Top genres added to MusicBrainz ──
         discogs_genre      n
            Electronic 542800
                  Rock 524578
                  Jazz 120143
                   Pop 112657
Folk, World, & Country  78343
               Hip Hop  75004
             Classical  46101
           Funk / Soul  29231
                 Latin  25702
                Reggae  15156
                 Blues  13707
             Non-Music  12079
        Stage & Screen   9542
            Children's   2844
      Brass & Military    704
                     1      4

── 3-source corroboration (CDS + WDC + Discogs) ──
 discogs_enriched  also_in_cds  also_in_wdc  in_all_3_sources
          1608595         2683         9937           

,discogs_artist,discogs_title,discogs_genre,discogs_style,discogs_country,discogs_year,discogs_label,release_id
0,Tori Amos,Under The Pink,Rock,Alternative Rock,US,1994,Atlantic,2
1,Tori Amos,Boys For Pele,Rock,Alternative Rock,US,1996,Atlantic,3
2,Portishead,Dummy,Electronic,Trip Hop,Canada,1995,Go! Discs,5
3,Tori Amos,To Venus And Back,Rock,Alternative Rock,US,1999,Atlantic,6
4,Tori Amos,Strange Little Girls,Rock,Alternative Rock,US,2001,Atlantic,7
5,Tori Amos,A Sorta Fairytale,Rock,Alternative Rock,Europe,2002,Epic,9
6,Tori Amos,Glory Of The 80's,Rock,Alternative Rock,Europe,1999,Atlantic,11
7,Tori Amos,Silent All These Years,Rock,Art Rock,UK,1992,EastWest,12
8,Rage Against The Machine,Rage Against The Machine,Rock,Hardcore Hip-Hop,Europe,2001,SMV Enterprises,14
9,Super Furry Animals,Llanfairpwllgwyngyllgogerychwyndrobwllantysiliogogogochynygofod (In Space) EP,Rock,Indie Rock,UK,1995,Ankst,15


In [2]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

# ════════════════════════════════════════════════════════════════
# Build the final 3-source enriched table
#
# APPROACH: Extend musicbrainz_release_enriched_v2 by
# LEFT JOINing the Discogs delta (only 1.6M rows — manageable)
# onto the existing v2 table.
#
# This is safe because:
#   - v2 already has CDS + WDC enrichment attached
#   - discogs_mb_enrichment_delta is only 1.6M rows
#   - LEFT JOIN preserves all 5.3M MB releases
#   - Discogs columns = NULL for unmatched releases
# ════════════════════════════════════════════════════════════════
print("Building musicbrainz_release_enriched_v3 ...")
print("= v2 (CDS + WDC) + Discogs enrichment layer\n")

con.execute("""
    CREATE OR REPLACE TABLE musicbrainz_release_enriched_v3 AS
    SELECT
        -- ── All columns from v2 (CDS + WDC enrichment) ──────
        v2.base_record_id,
        v2.release_id,
        v2.release_title,
        v2.main_artist_name,
        v2.artist_credit_full,
        v2.barcode,
        v2.mb_year,
        v2.total_tracks,
        v2.base_record_status,
        v2.is_linked_to_musicbrainz,

        -- CDS enrichment (Phase 1)
        v2.cds_category,
        v2.cds_genre,
        v2.cds_enrichment_source,
        v2.cds_enrichment_status,

        -- WebTables enrichment (Phase 2)
        v2.wt_genre,
        v2.wt_label,
        v2.wt_year,
        v2.wt_source_url,
        v2.wt_domain,
        v2.wt_match_type,
        v2.wt_match_score,

        -- Enrichment flags from v2
        v2.is_enriched_from_cds,
        v2.is_enriched_from_wt,
        v2.is_corroborated        AS is_corroborated_cds_wt,

        -- ── NEW: Discogs enrichment (Phase 3) ───────────────
        d.discogs_id,
        d.discogs_genre,
        d.discogs_style,          -- NEW field not in any other source
        d.discogs_label,
        d.discogs_catno,          -- NEW field not in any other source
        d.discogs_format,         -- NEW field not in any other source
        d.discogs_country,        -- NEW field not in any other source
        d.discogs_year,

        -- Discogs enrichment flag
        (d.release_id IS NOT NULL) AS is_enriched_from_discogs,

        -- ── Combined enrichment flags ────────────────────────
        -- Enriched by at least one source
        (v2.is_enriched_from_cds = TRUE
         OR v2.is_enriched_from_wt = TRUE
         OR d.release_id IS NOT NULL)   AS is_enriched_any,

        -- 3-way corroboration: confirmed by ALL THREE sources
        (v2.is_enriched_from_cds = TRUE
         AND v2.is_enriched_from_wt = TRUE
         AND d.release_id IS NOT NULL)  AS is_corroborated_all_3,

        -- 2-way: CDS + Discogs
        (v2.is_enriched_from_cds = TRUE
         AND d.release_id IS NOT NULL)  AS is_corroborated_cds_discogs,

        -- 2-way: WDC + Discogs
        (v2.is_enriched_from_wt = TRUE
         AND d.release_id IS NOT NULL)  AS is_corroborated_wt_discogs,

        -- ── Best available genre (priority: CDS > Discogs > WDC)
        COALESCE(
            NULLIF(CAST(v2.cds_genre AS VARCHAR), 'None'),
            d.discogs_genre,
            v2.wt_genre
        )                              AS best_genre,

        -- ── Best available label (priority: WDC > Discogs)
        COALESCE(
            v2.wt_label,
            d.discogs_label
        )                              AS best_label

    FROM musicbrainz_release_enriched_v2 v2
    LEFT JOIN discogs_mb_enrichment_delta d
      ON v2.release_id = d.release_id
""")

n_total = con.execute(
    "SELECT COUNT(*) FROM musicbrainz_release_enriched_v3"
).fetchone()[0]
print(f"✅ musicbrainz_release_enriched_v3: {n_total:,} rows")

Connected ✅
Building musicbrainz_release_enriched_v3 ...
= v2 (CDS + WDC) + Discogs enrichment layer



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ musicbrainz_release_enriched_v3: 5,314,525 rows


In [3]:
print("\n" + "═"*65)
print("  FINAL THESIS REPORT — 3-SOURCE ENRICHMENT COMPLETE")
print("═"*65)

stats = con.execute("""
    SELECT
        COUNT(*)                                        AS total,

        -- Phase 1: CDS
        COUNT(CASE WHEN is_enriched_from_cds = TRUE
                   THEN 1 END)                          AS cds_count,

        -- Phase 2: WebTables
        COUNT(CASE WHEN is_enriched_from_wt = TRUE
                   THEN 1 END)                          AS wt_count,

        -- Phase 3: Discogs
        COUNT(CASE WHEN is_enriched_from_discogs = TRUE
                   THEN 1 END)                          AS discogs_count,

        -- Enriched by at least one
        COUNT(CASE WHEN is_enriched_any = TRUE
                   THEN 1 END)                          AS any_count,

        -- Not enriched
        COUNT(CASE WHEN is_enriched_any = FALSE
                   THEN 1 END)                          AS none_count,

        -- 2-way corroboration
        COUNT(CASE WHEN is_corroborated_cds_wt = TRUE
                   THEN 1 END)                          AS corr_cds_wt,
        COUNT(CASE WHEN is_corroborated_cds_discogs = TRUE
                   THEN 1 END)                          AS corr_cds_disc,
        COUNT(CASE WHEN is_corroborated_wt_discogs = TRUE
                   THEN 1 END)                          AS corr_wt_disc,

        -- 3-way corroboration
        COUNT(CASE WHEN is_corroborated_all_3 = TRUE
                   THEN 1 END)                          AS corr_all_3

    FROM musicbrainz_release_enriched_v3
    WHERE release_id IS NOT NULL
""").fetchone()

total    = stats[0]
cds      = stats[1]
wt       = stats[2]
disc     = stats[3]
any_src  = stats[4]
none_src = stats[5]
c_cw     = stats[6]
c_cd     = stats[7]
c_wd     = stats[8]
c_all3   = stats[9]

print(f"""
  MusicBrainz total releases       : {total:>10,}
  ───────────────────────────────────────────────────────
  Phase 1 — CDS enrichment         : {cds:>10,}  ({100*cds/total:.4f}%)
  Phase 2 — WebTables enrichment   : {wt:>10,}  ({100*wt/total:.4f}%)
  Phase 3 — Discogs enrichment     : {disc:>10,}  ({100*disc/total:.4f}%)
  ───────────────────────────────────────────────────────
  Enriched by at least 1 source    : {any_src:>10,}  ({100*any_src/total:.4f}%)
  Still not enriched               : {none_src:>10,}  ({100*none_src/total:.4f}%)
  ───────────────────────────────────────────────────────
  2-way: CDS  + WebTables          : {c_cw:>10,}
  2-way: CDS  + Discogs            : {c_cd:>10,}
  2-way: WDC  + Discogs            : {c_wd:>10,}
  3-way: ALL THREE sources         : {c_all3:>10,}  ← highest confidence
  ───────────────────────────────────────────────────────
""")
print("═"*65)


═════════════════════════════════════════════════════════════════
  FINAL THESIS REPORT — 3-SOURCE ENRICHMENT COMPLETE
═════════════════════════════════════════════════════════════════

  MusicBrainz total releases       :  5,309,878
  ───────────────────────────────────────────────────────
  Phase 1 — CDS enrichment         :      4,200  (0.0791%)
  Phase 2 — WebTables enrichment   :     15,474  (0.2914%)
  Phase 3 — Discogs enrichment     :  1,608,595  (30.2944%)
  ───────────────────────────────────────────────────────
  Enriched by at least 1 source    :  1,615,603  (30.4264%)
  Still not enriched               :  3,694,275  (69.5736%)
  ───────────────────────────────────────────────────────
  2-way: CDS  + WebTables          :        182
  2-way: CDS  + Discogs            :      2,683
  2-way: WDC  + Discogs            :      9,937
  3-way: ALL THREE sources         :        136  ← highest confidence
  ───────────────────────────────────────────────────────

════════════════════

In [4]:
print("── New fields added to MusicBrainz by each source ──\n")

print("CDS unique contributions:")
print(con.execute("""
    SELECT
        COUNT(CASE WHEN cds_genre IS NOT NULL
                    AND cds_genre NOT IN ('None','')
                   THEN 1 END)  AS releases_with_cds_genre,
        COUNT(CASE WHEN cds_category IS NOT NULL
                    AND cds_category NOT IN ('None','')
                   THEN 1 END)  AS releases_with_cds_category
    FROM musicbrainz_release_enriched_v3
    WHERE release_id IS NOT NULL
""").fetchdf().to_string(index=False))

print("\nWebTables unique contributions:")
print(con.execute("""
    SELECT
        COUNT(CASE WHEN wt_label IS NOT NULL
                   THEN 1 END)      AS releases_with_wt_label,
        COUNT(CASE WHEN wt_genre IS NOT NULL
                   THEN 1 END)      AS releases_with_wt_genre,
        COUNT(CASE WHEN wt_source_url IS NOT NULL
                   THEN 1 END)      AS releases_with_provenance_url
    FROM musicbrainz_release_enriched_v3
    WHERE release_id IS NOT NULL
""").fetchdf().to_string(index=False))

print("\nDiscogs unique contributions (fields not available elsewhere):")
print(con.execute("""
    SELECT
        COUNT(CASE WHEN discogs_genre    IS NOT NULL
                   THEN 1 END)          AS releases_with_genre,
        COUNT(CASE WHEN discogs_style    IS NOT NULL
                   THEN 1 END)          AS releases_with_style,
        COUNT(CASE WHEN discogs_country  IS NOT NULL
                   THEN 1 END)          AS releases_with_country,
        COUNT(CASE WHEN discogs_format   IS NOT NULL
                   THEN 1 END)          AS releases_with_format,
        COUNT(CASE WHEN discogs_catno    IS NOT NULL
                   THEN 1 END)          AS releases_with_catno
    FROM musicbrainz_release_enriched_v3
    WHERE release_id IS NOT NULL
""").fetchdf().to_string(index=False))

print("\n── Best genre coverage (COALESCE of all 3 sources) ──")
print(con.execute("""
    SELECT
        COUNT(CASE WHEN best_genre IS NOT NULL
                   THEN 1 END)                          AS has_best_genre,
        ROUND(100.0 *
            COUNT(CASE WHEN best_genre IS NOT NULL
                       THEN 1 END) / COUNT(*), 2)       AS pct
    FROM musicbrainz_release_enriched_v3
    WHERE release_id IS NOT NULL
""").fetchdf().to_string(index=False))

print("\n── Sample: 3-way corroborated releases ──")
display(con.execute("""
    SELECT
        main_artist_name,
        release_title,
        cds_genre,
        discogs_genre,
        discogs_style,
        wt_genre,
        discogs_country,
        discogs_label,
        best_genre,
        release_id
    FROM musicbrainz_release_enriched_v3
    WHERE is_corroborated_all_3 = TRUE
    LIMIT 15
""").fetchdf())

── New fields added to MusicBrainz by each source ──

CDS unique contributions:
 releases_with_cds_genre  releases_with_cds_category
                    2714                        4200

WebTables unique contributions:
 releases_with_wt_label  releases_with_wt_genre  releases_with_provenance_url
                   8700                      97                         15474

Discogs unique contributions (fields not available elsewhere):
 releases_with_genre  releases_with_style  releases_with_country  releases_with_format  releases_with_catno
             1608595              1436481                1555626               1608595              1608592

── Best genre coverage (COALESCE of all 3 sources) ──
 has_best_genre   pct
        1609639 30.31

── Sample: 3-way corroborated releases ──


,main_artist_name,release_title,cds_genre,discogs_genre,discogs_style,wt_genre,discogs_country,discogs_label,best_genre,release_id
0,Erykah Badu,Baduizm,jazz funk,Funk / Soul,Neo Soul,None,US,Kedar Entertainment,jazz funk,93489
1,The Herbaliser,Something Wicked This Way Comes,hip hop,Electronic,Trip Hop,None,UK,Ninja Tune,hip hop,38775
2,Pixies,Doolittle,other,Rock,Indie Rock,None,UK,4AD,other,3082
3,St. Johnny,Speed Is Dreaming,None,Rock,Indie Rock,None,Europe,Geffen Records,Rock,35470
4,Ugly Kid Joe,America’s Least Wanted,None,Rock,Hard Rock,None,Canada,Stardog Records,Rock,67520
5,Billy Cobham,Crosswinds,jazz,Jazz,Fusion,None,US,Atlantic,jazz,4733
6,Radiohead,Hail to the Thief,None,Electronic,Alternative Rock,None,Europe,Parlophone,Electronic,1780
7,Ismaël Lô,Jammu Africa,tribal,"Folk, World, & Country",African,None,US,Triloka Records,tribal,18317
8,Massive Attack,Mezzanine,None,Electronic,Leftfield,None,Europe,Circa,Electronic,8067
9,Teenage Fanclub,Songs From Northern Britain,indie,Rock,Indie Rock,None,US,Columbia,indie,2148


In [5]:
import os

out_dir = "../data_processed/exports"
os.makedirs(out_dir, exist_ok=True)

# Export enriched-only subset (manageable size)
print("Exporting enriched releases to CSV ...")
con.execute(f"""
    COPY (
        SELECT *
        FROM musicbrainz_release_enriched_v3
        WHERE is_enriched_any = TRUE
          AND release_id IS NOT NULL
    )
    TO '{out_dir}/musicbrainz_enriched_v3_subset.csv'
    WITH (HEADER, DELIMITER ',')
""")

n_exported = con.execute("""
    SELECT COUNT(*) FROM musicbrainz_release_enriched_v3
    WHERE is_enriched_any = TRUE
      AND release_id IS NOT NULL
""").fetchone()[0]

size_mb = os.path.getsize(
    f"{out_dir}/musicbrainz_enriched_v3_subset.csv"
) / (1024**2)

print(f"✅ Exported: {n_exported:,} enriched releases")
print(f"📁 File: musicbrainz_enriched_v3_subset.csv")
print(f"📦 Size: {size_mb:.1f} MB")

print("\n── All pipeline tables summary ──")
tables = [
    "discogs_releases_raw",
    "discogs_cleaned_for_matching",
    "discogs_mb_exact_matches",
    "discogs_mb_enrichment_delta",
    "musicbrainz_release_enriched_v3"
]
for t in tables:
    try:
        n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
        print(f"  ✅  {t:<45} : {n:>12,}")
    except Exception as e:
        print(f"  ❌  {t:<45} : {e}")

Exporting enriched releases to CSV ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Exported: 1,615,603 enriched releases
📁 File: musicbrainz_enriched_v3_subset.csv
📦 Size: 394.7 MB

── All pipeline tables summary ──
  ✅  discogs_releases_raw                          :   19,035,253
  ✅  discogs_cleaned_for_matching                  :   18,356,215
  ✅  discogs_mb_exact_matches                      :    5,834,425
  ✅  discogs_mb_enrichment_delta                   :    1,608,595
  ✅  musicbrainz_release_enriched_v3               :    5,314,525


In [6]:
import duckdb
import os
import time

con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

out_dir = "../data_processed/exports"
os.makedirs(out_dir, exist_ok=True)

tables = [
    ("discogs_releases_raw",          "discogs_releases_raw.csv"),
    ("discogs_cleaned_for_matching",  "discogs_cleaned_for_matching.csv"),
    ("discogs_mb_exact_matches",      "discogs_mb_exact_matches.csv"),
    ("discogs_mb_enrichment_delta",   "discogs_mb_enrichment_delta.csv"),
]

print(f"Exporting {len(tables)} tables to {out_dir}\n")

for table, filename in tables:
    path = os.path.join(out_dir, filename)
    start = time.time()

    print(f"Exporting {table} ...")
    con.execute(f"""
        COPY {table}
        TO '{path}'
        WITH (HEADER, DELIMITER ',')
    """)

    elapsed  = time.time() - start
    size_mb  = os.path.getsize(path) / (1024**2)
    n_rows   = con.execute(f"SELECT COUNT(*) FROM {table}").fetchone()[0]

    print(f"  ✅ {filename}")
    print(f"     Rows : {n_rows:,}")
    print(f"     Size : {size_mb:,.1f} MB")
    print(f"     Time : {elapsed:.0f}s\n")

# ── Summary ──────────────────────────────────────────────────────
print("── Export summary ──")
total_size = 0
for _, filename in tables:
    path     = os.path.join(out_dir, filename)
    size_mb  = os.path.getsize(path) / (1024**2)
    total_size += size_mb
    print(f"  {filename:<45} : {size_mb:>8,.1f} MB")

print(f"\n  Total size : {total_size:,.1f} MB")
print(f"  Location   : {os.path.abspath(out_dir)}")
print("\n✅ All exports complete")

Connected ✅
Exporting 4 tables to ../data_processed/exports

Exporting discogs_releases_raw ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✅ discogs_releases_raw.csv
     Rows : 19,035,253
     Size : 2,021.9 MB
     Time : 20s

Exporting discogs_cleaned_for_matching ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✅ discogs_cleaned_for_matching.csv
     Rows : 18,356,215
     Size : 2,387.8 MB
     Time : 34s

Exporting discogs_mb_exact_matches ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✅ discogs_mb_exact_matches.csv
     Rows : 5,834,425
     Size : 934.3 MB
     Time : 14s

Exporting discogs_mb_enrichment_delta ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  ✅ discogs_mb_enrichment_delta.csv
     Rows : 1,608,595
     Size : 181.6 MB
     Time : 3s

── Export summary ──
  discogs_releases_raw.csv                      :  2,021.9 MB
  discogs_cleaned_for_matching.csv              :  2,387.8 MB
  discogs_mb_exact_matches.csv                  :    934.3 MB
  discogs_mb_enrichment_delta.csv               :    181.6 MB

  Total size : 5,525.5 MB
  Location   : C:\Users\User\OneDrive\Desktop\Eman_Thesis\data_processed\exports

✅ All exports complete


In [7]:
import duckdb
import pandas as pd

pd.set_option('display.max_colwidth', None)
con = duckdb.connect("../data_processed/thesis.duckdb")
print("Connected ✅")

print("── musicbrainz_release_enriched_v3 columns ──")
print(con.execute("""
    SELECT column_name, data_type
    FROM information_schema.columns
    WHERE table_name = 'musicbrainz_release_enriched_v3'
    ORDER BY ordinal_position
""").fetchdf().to_string(index=False))

print("\n── Row count ──")
n = con.execute("""
    SELECT COUNT(*) FROM musicbrainz_release_enriched_v3
""").fetchone()[0]
print(f"Total rows: {n:,}")

print("\n── Sample row showing ALL 3 sources ──")
display(con.execute("""
    SELECT
        release_id,
        main_artist_name,
        release_title,
        mb_year,

        -- CDS fields
        cds_genre,
        cds_category,

        -- WebTables fields
        wt_genre,
        wt_label,
        wt_source_url,

        -- Discogs fields
        discogs_genre,
        discogs_style,
        discogs_label,
        discogs_country,
        discogs_format,
        discogs_catno,
        discogs_year,

        -- Combined best fields
        best_genre,
        best_label,

        -- Enrichment flags
        is_enriched_from_cds,
        is_enriched_from_wt,
        is_enriched_from_discogs,
        is_enriched_any,
        is_corroborated_all_3

    FROM musicbrainz_release_enriched_v3
    WHERE is_corroborated_all_3 = TRUE
    LIMIT 5
""").fetchdf())

Connected ✅
── musicbrainz_release_enriched_v3 columns ──
                column_name data_type
             base_record_id    BIGINT
                 release_id    BIGINT
              release_title   VARCHAR
           main_artist_name   VARCHAR
         artist_credit_full   VARCHAR
                    barcode   VARCHAR
                    mb_year    BIGINT
               total_tracks   HUGEINT
         base_record_status   VARCHAR
   is_linked_to_musicbrainz   BOOLEAN
               cds_category   VARCHAR
                  cds_genre   VARCHAR
      cds_enrichment_source   VARCHAR
      cds_enrichment_status   VARCHAR
                   wt_genre   VARCHAR
                   wt_label   VARCHAR
                    wt_year   VARCHAR
              wt_source_url   VARCHAR
                  wt_domain   VARCHAR
              wt_match_type   VARCHAR
             wt_match_score    DOUBLE
       is_enriched_from_cds   BOOLEAN
        is_enriched_from_wt   BOOLEAN
     is_corroborated_cds_wt   

,release_id,main_artist_name,release_title,mb_year,cds_genre,cds_category,wt_genre,wt_label,wt_source_url,discogs_genre,...,discogs_format,discogs_catno,discogs_year,best_genre,best_label,is_enriched_from_cds,is_enriched_from_wt,is_enriched_from_discogs,is_enriched_any,is_corroborated_all_3
0,93489,Erykah Badu,Baduizm,2025,jazz funk,jazz,None,Kedar Entertainment,http://kfai.org/node/33760?page=0%2C0,Funk / Soul,...,Vinyl,U-53027,1997,jazz funk,Kedar Entertainment,True,True,True,True,True
1,38775,The Herbaliser,Something Wicked This Way Comes,2013,hip hop,rock,None,Ninja Tune 10:29,http://kboo.org/playlists/wp-content/themes/echoes/%3A37?page=4,Electronic,...,CD,ZEN CD64,2002,hip hop,Ninja Tune 10:29,True,True,True,True,True
2,3082,Pixies,Doolittle,2015,other,misc,None,Matador,http://kfai.org/node/19691?page=0%2C1,Rock,...,Cassette,CAD C 905,1989,other,Matador,True,True,True,True,True
3,35470,St. Johnny,Speed Is Dreaming,2012,None,rock,None,Geffen Records,http://www.wnur.org/programming/playlist/?date=2014-02-14&time=4:00pm,Rock,...,Vinyl,GEF-24534,1994,Rock,Geffen Records,True,True,True,True,True
4,67520,Ugly Kid Joe,America’s Least Wanted,2026,None,rock,None,None,http://www.aria.com.au/pages/aria-charts-end-of-year-charts-top-50albums-1993.htm,Rock,...,CD,314 512 571-2,1992,Rock,Stardog Records,True,True,True,True,True


In [9]:
import os
import time

out_dir = "../data_processed/exports"
os.makedirs(out_dir, exist_ok=True)

# ── Option A: Full table (5.3M rows) ────────────────────────────
print("Exporting full musicbrainz_release_enriched_v3 ...")
print("Warning: this will be ~2-3 GB\n")

start = time.time()
con.execute(f"""
    COPY musicbrainz_release_enriched_v3
    TO '{out_dir}/musicbrainz_release_enriched_v3_FULL.csv'
    WITH (HEADER, DELIMITER ',')
""")
elapsed = time.time() - start
size_mb = os.path.getsize(
    f"{out_dir}/musicbrainz_release_enriched_v3_FULL.csv"
) / (1024**2)
print(f"✅ Full table exported")
print(f"   Rows : 5,314,525")
print(f"   Size : {size_mb:,.1f} MB")
print(f"   Time : {elapsed:.0f}s")

# ── Option B: Enriched only subset (1.6M rows) ──────────────────
print("\nExporting enriched-only subset ...")

start = time.time()
con.execute(f"""
    COPY (
        SELECT *
        FROM musicbrainz_release_enriched_v3
        WHERE is_enriched_any = TRUE
          AND release_id IS NOT NULL
        ORDER BY release_id
    )
    TO '{out_dir}/musicbrainz_enriched_v3_subset.csv'
    WITH (HEADER, DELIMITER ',')
""")
elapsed = time.time() - start
size_mb = os.path.getsize(
    f"{out_dir}/musicbrainz_enriched_v3_subset.csv"
) / (1024**2)
n = con.execute("""
    SELECT COUNT(*) FROM musicbrainz_release_enriched_v3
    WHERE is_enriched_any = TRUE
""").fetchone()[0]
print(f"✅ Enriched subset exported")
print(f"   Rows : {n:,}")
print(f"   Size : {size_mb:,.1f} MB")
print(f"   Time : {elapsed:.0f}s")

# ── Option C: 3-way corroborated only (136 rows — tiny) ─────────
print("\nExporting 3-way corroborated releases ...")

con.execute(f"""
    COPY (
        SELECT *
        FROM musicbrainz_release_enriched_v3
        WHERE is_corroborated_all_3 = TRUE
    )
    TO '{out_dir}/musicbrainz_3way_corroborated.csv'
    WITH (HEADER, DELIMITER ',')
""")
size_kb = os.path.getsize(
    f"{out_dir}/musicbrainz_3way_corroborated.csv"
) / 1024
print(f"✅ 3-way corroborated exported")
print(f"   Rows : 136")
print(f"   Size : {size_kb:.1f} KB")

# ── Summary ──────────────────────────────────────────────────────
print("\n" + "═"*60)
print("  EXPORT SUMMARY")
print("═"*60)

files = [
    "musicbrainz_release_enriched_v3_FULL.csv",
    "musicbrainz_enriched_v3_subset.csv",
    "musicbrainz_3way_corroborated.csv"
]

total = 0
for f in files:
    path = os.path.join(out_dir, f)
    if os.path.exists(path):
        mb = os.path.getsize(path) / (1024**2)
        total += mb
        print(f"  {f:<50} : {mb:>8,.1f} MB")

print(f"\n  Total size : {total:,.1f} MB")
print(f"  Location   : {os.path.abspath(out_dir)}")

Exporting full musicbrainz_release_enriched_v3 ...



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Full table exported
   Rows : 5,314,525
   Size : 1,032.7 MB
   Time : 9s

Exporting enriched-only subset ...


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Enriched subset exported
   Rows : 1,615,603
   Size : 394.7 MB
   Time : 11s

Exporting 3-way corroborated releases ...
✅ 3-way corroborated exported
   Rows : 136
   Size : 47.4 KB

════════════════════════════════════════════════════════════
  EXPORT SUMMARY
════════════════════════════════════════════════════════════
  musicbrainz_release_enriched_v3_FULL.csv           :  1,032.7 MB
  musicbrainz_enriched_v3_subset.csv                 :    394.7 MB
  musicbrainz_3way_corroborated.csv                  :      0.0 MB

  Total size : 1,427.5 MB
  Location   : C:\Users\User\OneDrive\Desktop\Eman_Thesis\data_processed\exports
